In [1]:
import pandas as pd
import numpy as np
import openpyxl
import re
import os

# ══════════════════════════════════════════════════════════════════
# COMPLETE DATA WRANGLING SCRIPT
# Covers: Loading, Cleaning (C1-C12), Merging, 
#         Final Transformations, Export
# ══════════════════════════════════════════════════════════════════

# ── LOAD ALL RAW DATASETS ─────────────────────────────────────────

hh    = pd.read_csv("household_vista_2024_2025.csv")
pers  = pd.read_csv("person_vista_2024_2025.csv")
trips = pd.read_csv("trips_vista_2024_2025.csv")
budget = pd.read_csv("state-budget-2023-24-budget-website-map-data.csv")

wb = openpyxl.load_workbook("Local Government Area, Indexes, SEIFA 2021.xlsx", read_only=True)
ws = wb["Table 1"]
rows = []
for i, row in enumerate(ws.iter_rows(values_only=True)):
    if i < 6: continue
    if row[0] is None: break
    rows.append(row[:11])
seifa = pd.DataFrame(rows, columns=[
    "LGA_Code","LGA_Name","IRSD_Score","IRSD_Decile",
    "IRSAD_Score","IRSAD_Decile","IER_Score","IER_Decile",
    "IEO_Score","IEO_Decile","Population"])
seifa["LGA_Code"] = seifa["LGA_Code"].astype(str)
vic_seifa = seifa[seifa["LGA_Code"].str.startswith("2")].copy()

print("All datasets loaded.")
print(f"hh={hh.shape}, pers={pers.shape}, trips={trips.shape}, "
      f"budget={budget.shape}, vic_seifa={vic_seifa.shape}")

# ══════════════════════════════════════════════════════════════════
# HOUSEHOLD CLEANING
# ══════════════════════════════════════════════════════════════════

# C1: hhinc_group — fill NaN with "Not Stated"
# Reason: 73 genuine survey refusals. Retaining rows for travel
# analysis; "Not Stated" keeps them visible in groupby operations.
hh['hhinc_group'] = hh['hhinc_group'].fillna("Not Stated")
print(f"\nC1: hhinc_group NaN → 'Not Stated': "
      f"{(hh['hhinc_group']=='Not Stated').sum()} rows")

# C2: totalbikes — convert to numeric, recode survey refusals as NaN
# Reason: Column stored as string due to "Missing/Refused" survey code.
hh['totalbikes_clean'] = pd.to_numeric(hh['totalbikes'], errors='coerce')
print(f"C2: totalbikes 'Missing/Refused' → NaN: "
      f"{hh['totalbikes_clean'].isna().sum()} NaNs "
      f"(range {hh['totalbikes_clean'].min():.0f}–"
      f"{hh['totalbikes_clean'].max():.0f})")

# C3: LGA name standardisation for household home LGA
# Reason: VISTA uses "(C)", "(S)", "(B)" suffixes; SEIFA and Budget
# do not. Three additional mismatches require a manual mapping:
# - Merri-bek renamed from Moreland in Nov 2022 (SEIFA predates rename)
# - Bayside and Kingston include "(Vic.)" suffix in SEIFA 2021 to
#   distinguish from same-named LGAs in other states.
def strip_lga_suffix(name):
    """Remove council type suffixes: (C), (S), (B), (RC) etc."""
    return re.sub(r'\s*\([^)]*\)', '', str(name)).strip()

hh['lga_standard'] = hh['homelga'].apply(strip_lga_suffix)

seifa_name_map = {
    'Bayside':   'Bayside (Vic.)',
    'Kingston':  'Kingston (Vic.)',
    'Merri-bek': 'Moreland'
}
hh['lga_for_seifa'] = hh['lga_standard'].replace(seifa_name_map)

seifa_names = set(vic_seifa['LGA_Name'].str.strip())
unmatched = set(hh['lga_for_seifa'].unique()) - seifa_names
print(f"C3: LGA standardised. Unmatched after fix: {unmatched}")

# ══════════════════════════════════════════════════════════════════
# PERSONS CLEANING
# ══════════════════════════════════════════════════════════════════

# C4: numstops — convert to numeric, recode refusals as NaN
# Reason: 60 survey refusals stored as text string.
pers['numstops_clean'] = pd.to_numeric(pers['numstops'], errors='coerce')
print(f"\nC4: numstops refusals → NaN: "
      f"{pers['numstops_clean'].isna().sum()} NaNs")

# Note: 60 persons have missing perspoststratweight.
# These persons recorded zero trips and are children/non-respondents.
# Weights are not used in this EDA so no action required.
print(f"NOTE: {pers['perspoststratweight'].isna().sum()} persons with "
      f"missing weights — all have 0 trips recorded. Retained as-is.")

# ══════════════════════════════════════════════════════════════════
# TRIPS CLEANING
# ══════════════════════════════════════════════════════════════════

# C5: starthour normalisation — map hours 24-27 to 0-3
# Reason: VISTA records overnight trips using hours > 23 continuing
# from the previous day (e.g. 25 = 1am next day). This is correct
# VISTA convention but causes axis distortion in time-of-day charts.
trips['starthour_norm'] = trips['starthour'].apply(
    lambda h: h - 24 if h > 23 else h)
print(f"\nC5: starthour_norm created. "
      f"Trips with hour > 23: {(trips['starthour'] > 23).sum()} "
      f"(mapped to 0–3 in starthour_norm)")

# C6: cumdist and dist1-dist9 — convert string to float (km)
# Reason: Mostly numeric strings. 104 rows contain "Missing" survey
# code — converted to NaN. Mean 9.7km, max 381km is valid range
# for Melbourne metropolitan trips.
trips['cumdist_km'] = pd.to_numeric(trips['cumdist'], errors='coerce')
for col in [f'dist{i}' for i in range(1, 10)]:
    trips[f'{col}_km'] = pd.to_numeric(trips[col], errors='coerce')
print(f"C6: cumdist_km range: "
      f"{trips['cumdist_km'].min():.3f}–{trips['cumdist_km'].max():.3f} km. "
      f"NaNs: {trips['cumdist_km'].isna().sum()} (coded 'Missing' in source)")

# C7: time2-time9 — convert string to numeric (minutes)
for col in [f'time{i}' for i in range(2, 10)]:
    trips[f'{col}_min'] = pd.to_numeric(trips[col], errors='coerce')
print(f"C7: time2-time9 converted to numeric (minutes).")

# C8: LGA name standardisation in trips origlga and destlga
# Reason: Applies same suffix stripping as households.
# Non-Victorian LGAs (Ballarat, Interstate etc.) are legitimate
# trip origins/destinations — retained as-is.
trips['origlga_std'] = trips['origlga'].apply(strip_lga_suffix)
trips['destlga_std'] = trips['destlga'].apply(strip_lga_suffix)
print(f"C8: origlga_std/destlga_std standardised.")

# C9: Trip time outlier flag
# Reason: 5 trips exceed 480 minutes. Four are vehicle driver trips
# within metro Melbourne which is implausible as a single trip.
# One is an interstate flight (plausible). Flagged but retained —
# they represent 0.025% of data and won't affect aggregate analysis.
trips['triptime_flag'] = trips['triptime'] > 480
print(f"C9: {trips['triptime_flag'].sum()} trips flagged as outliers (>480 min)")

# Note on 'duration' column: Records time spent AT the destination
# (activity duration), not travel time. Values of 999 indicate end
# of travel diary day. NOT used in analysis — triptime is the
# correct travel time measure.

# ══════════════════════════════════════════════════════════════════
# BUDGET CLEANING
# ══════════════════════════════════════════════════════════════════

# C10: Rename investment column (has trailing space in source data)
budget = budget.rename(columns={'investment ': 'investment'})
print(f"\nC10: Budget 'investment ' renamed to 'investment'")

# C11: Parse investment to numeric millions
# Reason: Stored as currency string e.g. "$42.2 million*"
# Asterisk (*) = bundled funding shared across multiple LGA rows.
# We use project COUNT not dollar sums for Q1/Q3 to avoid
# double-counting bundled investment values.
def parse_investment(val):
    if pd.isna(val): return np.nan
    val_str = str(val).strip().replace('*','').replace('$','').replace(',','').strip()
    if 'billion' in val_str.lower():
        num = float(re.sub(r'[^\d.]', '', val_str.lower().replace('billion','')))
        return num * 1000
    elif 'million' in val_str.lower():
        num = float(re.sub(r'[^\d.]', '', val_str.lower().replace('million','')))
        return num
    else:
        num = float(re.sub(r'[^\d.]', '', val_str))
        return num / 1_000_000

budget['investment_millions'] = budget['investment'].apply(parse_investment)
budget['investment_bundled']  = budget['investment'].str.strip()\
    .str.endswith('*').fillna(False)
print(f"C11: investment parsed. Range: "
      f"${budget['investment_millions'].min():.3f}M–"
      f"${budget['investment_millions'].max():.1f}M. "
      f"Bundled rows: {budget['investment_bundled'].sum()}")

# C12: Spatial flag — rows without coordinates are statewide programs
budget['has_location'] = budget['lat'].notna()
print(f"C12: Spatial flag. With location: {budget['has_location'].sum()}, "
      f"Without: {(~budget['has_location']).sum()}")

# ══════════════════════════════════════════════════════════════════
# DERIVED: Investment tier per LGA (for Q1 and Q3)
# Tier classification based on education project count per LGA.
# Project count used over dollar value to avoid double-counting
# bundled investment. Tiers based on natural distribution breaks:
# None=0, Low=1-2, Medium=3-5, High=6+ projects.
# ══════════════════════════════════════════════════════════════════

edu_located = budget[
    (budget['theme'] == 'Education') &
    (budget['has_location'] == True)
].copy()

lga_investment = edu_located.groupby('lga').agg(
    project_count=('unique_id', 'count'),
    project_types=('type', lambda x: list(x.unique()))
).reset_index()
lga_investment.rename(columns={'lga': 'lga_standard'}, inplace=True)

def assign_tier(n):
    if n == 0:   return 'None'
    elif n <= 2: return 'Low (1-2)'
    elif n <= 5: return 'Medium (3-5)'
    else:        return 'High (6+)'

lga_investment['investment_tier'] = \
    lga_investment['project_count'].apply(assign_tier)

# Add "None" tier for VISTA LGAs with zero education projects
all_vista_lgas = hh['lga_standard'].unique()
lgas_with_projects = set(lga_investment['lga_standard'])
none_rows = pd.DataFrame({
    'lga_standard':   [l for l in all_vista_lgas if l not in lgas_with_projects],
    'project_count':  0,
    'project_types':  [[] for _ in range(sum(l not in lgas_with_projects
                                             for l in all_vista_lgas))],
    'investment_tier':'None'
})
lga_investment = pd.concat([lga_investment, none_rows], ignore_index=True)

# Sort key: None=0, Low=1, Medium=2, High=3
tier_order = {'None': 0, 'Low (1-2)': 1, 'Medium (3-5)': 2, 'High (6+)': 3}
lga_investment['investment_tier_order'] = \
    lga_investment['investment_tier'].map(tier_order)

print(f"\nInvestment tier table ({len(lga_investment)} LGAs):")
print(lga_investment['investment_tier'].value_counts())

# ══════════════════════════════════════════════════════════════════
# MODE GROUP mapping (for Q1 and Q2 visualisations)
# Reason: 16 raw linkmode values produce unreadable charts.
# Grouped into 5 standard transport planning categories.
# ══════════════════════════════════════════════════════════════════

mode_group_map = {
    'Vehicle Driver':    'Private Vehicle',
    'Vehicle Passenger': 'Private Vehicle',
    'Motorcycle':        'Private Vehicle',
    'Train':             'Public Transport',
    'Tram':              'Public Transport',
    'Public Bus':        'Public Transport',
    'School Bus':        'Public Transport',
    'Walking':           'Active Transport',
    'Bicycle':           'Active Transport',
    'Running/jogging':   'Active Transport',
    'e-Scooter':         'Active Transport',
    'Rideshare Service': 'Ride-hailing',
    'Taxi':              'Ride-hailing',
    'Plane':             'Other',
    'Other':             'Other',
    'Mobility Scooter':  'Other'
}

# ══════════════════════════════════════════════════════════════════
# INCOME GROUP sort key
# Reason: hhinc_group sorts alphabetically in Tableau.
# Extract lower bound of weekly income range as numeric sort key.
# ══════════════════════════════════════════════════════════════════

def income_sort_key(val):
    if pd.isna(val) or val in ['Not Stated']: return -1
    match = re.search(r'\$(\d+(?:,\d+)?)-', val)
    if match: return int(match.group(1).replace(',', ''))
    return 0

# ══════════════════════════════════════════════════════════════════
# MERGE 1: Full VISTA merged dataset
# trips + persons + households + investment tier
# Note: homesubregion_ASGS, homeregion_ASGS, dayType already exist
# in trips — excluded from hh selection to avoid _x/_y suffixes.
# ══════════════════════════════════════════════════════════════════

vista_merged = trips.merge(
    pers[['persid','hhid','agegroup','sex','mainact',
          'anywork','anywfh','persinc','studying',
          'numstops_clean','perspoststratweight']],
    on=['persid','hhid'], how='left'
).merge(
    hh[['hhid','hhsize','totalvehs','totalbikes_clean',
        'hhinc_group','lga_standard','lga_for_seifa',
        'travmonth','travdow']],
    on='hhid', how='left'
).merge(
    lga_investment[['lga_standard','project_count',
                    'investment_tier','investment_tier_order']],
    on='lga_standard', how='left'
)

print(f"\nMERGE 1: vista_merged {vista_merged.shape}")
print(f"  Rows preserved: {len(vista_merged) == 19864}")

# ── Add SEIFA deciles to vista_merged (critical for Q2) ───────────
# Reason: Q2 requires IRSD_Decile on every trip row so Tableau
# can group trips by neighbourhood economic status without needing
# a separate data relationship join.
vista_merged = vista_merged.merge(
    vic_seifa[['LGA_Name','IRSD_Score','IRSD_Decile',
               'IEO_Score','IEO_Decile']],
    left_on='lga_for_seifa', right_on='LGA_Name', how='left'
)
print(f"  SEIFA added. IRSD_Decile nulls: "
      f"{vista_merged['IRSD_Decile'].isna().sum()}")

# ── Add mode_group and sort keys ──────────────────────────────────
vista_merged['mode_group'] = vista_merged['linkmode'].map(mode_group_map)
vista_merged['hhinc_sort'] = vista_merged['hhinc_group'].apply(income_sort_key)

print(f"  mode_group nulls: {vista_merged['mode_group'].isna().sum()}")
print(f"  Final shape: {vista_merged.shape}")

# ══════════════════════════════════════════════════════════════════
# MERGE 2: Households + SEIFA (for Q2)
# ══════════════════════════════════════════════════════════════════

hh_seifa = hh[['hhid','lga_standard','lga_for_seifa',
               'homesubregion_ASGS','homeregion_ASGS',
               'hhinc_group','totalvehs','totalbikes_clean',
               'hhsize','dayType','travmonth','travdow']].merge(
    vic_seifa[['LGA_Name','IRSD_Score','IRSD_Decile',
               'IRSAD_Score','IRSAD_Decile',
               'IEO_Score','IEO_Decile','Population']],
    left_on='lga_for_seifa', right_on='LGA_Name', how='left'
)
hh_seifa['hhinc_sort'] = hh_seifa['hhinc_group'].apply(income_sort_key)

print(f"\nMERGE 2: hh_seifa {hh_seifa.shape}")
print(f"  IRSD_Decile nulls: {hh_seifa['IRSD_Decile'].isna().sum()}")

# ══════════════════════════════════════════════════════════════════
# MERGE 3: Budget spatial education subset (for Q3)
# ══════════════════════════════════════════════════════════════════

budget_spatial = budget[
    (budget['has_location'] == True) &
    (budget['theme'] == 'Education')
].copy()
budget_spatial['lga_standard'] = budget_spatial['lga'].str.strip()

# LGA-level Q3 summary with VISTA coverage flag
vista_lgas = set(hh['lga_standard'].dropna().unique())
hh_per_lga = hh.groupby('lga_standard').size().reset_index(name='hh_count')

lga_q3 = budget_spatial.groupby('lga_standard').agg(
    project_count=('unique_id', 'count'),
    lat_mean=('lat', 'mean'),
    long_mean=('long', 'mean')
).reset_index().merge(
    lga_investment[['lga_standard','investment_tier',
                    'investment_tier_order']],
    on='lga_standard', how='left'
).merge(
    hh_per_lga, on='lga_standard', how='left'
)

lga_q3['in_vista']  = lga_q3['lga_standard'].isin(vista_lgas)
lga_q3['hh_count']  = lga_q3['hh_count'].fillna(0).astype(int)

print(f"\nMERGE 3: budget_spatial {budget_spatial.shape}, "
      f"lga_q3 {lga_q3.shape}")
print(f"  lga_q3 with VISTA data: {lga_q3['in_vista'].sum()}")
print(f"  lga_q3 without VISTA data: {(~lga_q3['in_vista']).sum()}")

# ══════════════════════════════════════════════════════════════════
# DERIVED: Work and Education trips subset (for Q1)
# ══════════════════════════════════════════════════════════════════

we_trips = vista_merged[
    vista_merged['trippurp'].isin(['Work Related','Education'])
].copy()

print(f"\nDERIVED: we_trips {we_trips.shape}")
print(f"  trippurp: {we_trips['trippurp'].value_counts().to_dict()}")

# ══════════════════════════════════════════════════════════════════
# FINAL VALIDATION
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

checks = {
    "vista_merged rows = 19864":
        len(vista_merged) == 19864,
    "hh_seifa rows = 2486":
        len(hh_seifa) == 2486,
    "we_trips rows = 4158":
        len(we_trips) == 4158,
    "vista_merged IRSD_Decile 0 nulls":
        vista_merged['IRSD_Decile'].isna().sum() == 0,
    "vista_merged mode_group 0 nulls":
        vista_merged['mode_group'].isna().sum() == 0,
    "vista_merged investment_tier 0 nulls":
        vista_merged['investment_tier'].isna().sum() == 0,
    "vista_merged trippurp 0 nulls":
        vista_merged['trippurp'].isna().sum() == 0,
    "vista_merged linkmode 0 nulls":
        vista_merged['linkmode'].isna().sum() == 0,
    "vista_merged lga_standard 0 nulls":
        vista_merged['lga_standard'].isna().sum() == 0,
    "hh_seifa IRSD_Decile 0 nulls":
        hh_seifa['IRSD_Decile'].isna().sum() == 0,
    "lga_q3 in_vista flag present":
        'in_vista' in lga_q3.columns,
    "lga_q3 hh_count 0 nulls":
        lga_q3['hh_count'].isna().sum() == 0,
    "budget_spatial has lat/long":
        budget_spatial['lat'].isna().sum() == 0,
}

all_pass = True
for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    if not result: all_pass = False
    print(f"  {status}: {check}")

print(f"\n{'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")

# ══════════════════════════════════════════════════════════════════
# EXPORT FINAL CSVs
# ══════════════════════════════════════════════════════════════════

vista_merged.to_csv("vista_merged_final.csv", index=False)
hh_seifa.to_csv("hh_seifa_final.csv", index=False)
budget_spatial.to_csv("budget_spatial_final.csv", index=False)
lga_q3.to_csv("lga_q3_final.csv", index=False)
lga_investment.to_csv("lga_investment_final.csv", index=False)
we_trips.to_csv("we_trips_final.csv", index=False)

print(f"\nExported files:")
for f in ["vista_merged_final.csv","hh_seifa_final.csv",
          "budget_spatial_final.csv","lga_q3_final.csv",
          "lga_investment_final.csv","we_trips_final.csv"]:
    size_mb = os.path.getsize(f) / 1_000_000
    rows = sum(1 for _ in open(f)) - 1
    print(f"  {f}: {rows} rows, {size_mb:.2f} MB")

print("\n" + "=" * 60)
print("WRANGLING COMPLETE — Ready for Tableau / R")
print("=" * 60)

All datasets loaded.
hh=(2486, 28), pers=(6404, 53), trips=(19864, 65), budget=(356, 11), vic_seifa=(80, 11)

C1: hhinc_group NaN → 'Not Stated': 73 rows
C2: totalbikes 'Missing/Refused' → NaN: 244 NaNs (range 0–12)
C3: LGA standardised. Unmatched after fix: set()

C4: numstops refusals → NaN: 60 NaNs
NOTE: 60 persons with missing weights — all have 0 trips recorded. Retained as-is.

C5: starthour_norm created. Trips with hour > 23: 80 (mapped to 0–3 in starthour_norm)
C6: cumdist_km range: 0.009–381.529 km. NaNs: 104 (coded 'Missing' in source)
C7: time2-time9 converted to numeric (minutes).
C8: origlga_std/destlga_std standardised.
C9: 5 trips flagged as outliers (>480 min)

C10: Budget 'investment ' renamed to 'investment'
C11: investment parsed. Range: $0.100M–$2800.0M. Bundled rows: 261
C12: Spatial flag. With location: 330, Without: 26

Investment tier table (67 LGAs):
investment_tier
Low (1-2)       36
Medium (3-5)    18
High (6+)        8
None             5
Name: count, dtype: 